<a href="https://colab.research.google.com/github/alimohmedelsaid26-cell/1000MLEngineer/blob/main/Household_Power_consumption_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn import metrics
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectFromModel
dataset = pd.read_csv('/content/household_power_consumption.txt.zip', sep=';', header=0, low_memory=False, infer_datetime_format=True, parse_dates={'datetime':[0,1]}, index_col=['datetime'])

/tmp/ipython-input-163/3796028250.py:11: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  dataset = pd.read_csv('/content/household_power_consumption.txt.zip', sep=';', header=0, low_memory=False, infer_datetime_format=True, parse_dates={'datetime':[0,1]}, index_col=['datetime'])
/tmp/ipython-input-163/3796028250.py:11: FutureWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  dataset = pd.read_csv('/content/household_power_consumption.txt.zip', sep=';', header=0, low_memory=False, infer_datetime_format=True, parse_dates={'datetime':[0,1]}, index_col=['datetime'])
/tmp/ipython-input-163/3796028250.py:11: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when day

In [ ]:

# 1. Clean missing value placeholders
dataset.replace('?', np.nan, inplace=True)
dataset.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 2. Convert columns to numeric properly
for col in dataset.columns:
    dataset[col] = pd.to_numeric(dataset[col], errors='coerce')

# 3. Handle Missing Values FIRST
# Using 'mean' is often better for power data than 'most_frequent'
imputer = SimpleImputer(strategy='mean')
dataset_cleansed = pd.DataFrame(imputer.fit_transform(dataset), columns=dataset.columns)

# 4. Feature Engineering (Calculating General Power)
# Ensure columns 0 and 1 are the ones you intend to square
v = dataset_cleansed.values
dataset_cleansed['General_Power'] = np.sqrt(np.square(v[:,0]) + np.square(v[:,1]))

# 5. Define X and y
X = dataset_cleansed.drop(columns=['General_Power'])
y = dataset_cleansed['General_Power']

# 6. Scaling
scaler_x = MinMaxScaler()
X_scaled = scaler_x.fit_transform(X)

scaler_y = MinMaxScaler()
# Reshape y to 2D for the scaler, then back to 1D
y_scaled = scaler_y.fit_transform(y.values.reshape(-1, 1)).ravel()

# 7. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.33, random_state=44)